# പാഠം 13 - കോഗ്നീ നോളജ് ഗ്രാഫുകളോടുള്ള ഏജന്റ് മെമ്മറി


## സെ tup

ഈ നോട്ട്ബുക്ക് [**Cognee**](https://www.cognee.ai/) നോളജ് ഗ്രാഫുകളും **Microsoft Agent Framework** (MAF) ഉപയോഗിച്ച് സ്ഥിരമായ മെമ്മറി ഉള്ള ഒരു അദ്ധ്വാനശീല **കോഡിംഗ് അസിസ്റ്റന്റ്** എങ്ങനെ നിർമ്മിക്കാമെന്ന് കാണിക്കുന്നു.

Cognee ഘടനയില്ലാത്ത ടെക്സ്റ്റ് ഘടനയുള്ള, ചോദ്യോദ്വേക വിഷയശേഷിയുള്ള ഡാറ്റാഗ്രാഫ്ഫിൽ (knowledge graph) മാറ്റുന്നു — ഇത് നിങ്ങളുടെ ഏജന്റിന് സമ്പന്നമായ, ബന്ധങ്ങളെ അറിയുന്ന ദീർഘകാല മെമ്മറി നൽകുന്നു.

### നിങ്ങൾ എന്ത് പഠിക്കും
1. **നോളജ് ഗ്രാഫുകൾ നിർമ്മിക്കുക**: വികസിപ്പകരുടെ പ്രൊഫൈലുകളും മികച്ച അനുഭവങ്ങളും ഘടനയുള്ള, ചോദ്യോദ്വേക പരിജ്ഞാനമായി മാറ്റുക.
2. **Cognee നെ MAF ഒപ്പം ലയിപ്പിക്കൽ**: `@tool` ഫംഗ്ഷനുകൾ ഉപയോഗിച്ച് ഒരു MAF ഏജന്റ് Cognee നോളജ് ഗ്രാഫിനെ ചോദ്യം ചെയ്യാൻ അനുവദിക്കുക.
3. **സെഷൻ-ജ്ഞാനമുള്ള സംവാദങ്ങൾ**: ഒരേ സെഷനിൽ നിരവധി ചോദ്യങ്ങളിൽ പരിപ്രേക്ഷ്യം നിലനിർത്തുക.
4. **ദീർഘകാല മെമ്മറി**: പ്രധാനപ്പെട്ട അറിവ് സെഷനുകൾക്ക് മുകളിലായിരിക്കാൻ നിലനിർത്തുകയും പുതിയ സംവാദങ്ങളിൽ തിരികെ ലഭ്യമാക്കുക.

### ആവശ്യമുള്ളതു
- Python 3.9+
- സെഷൻ മാനേജ്മെന്റിനായി ലോക്കലിൽ Redis ഓടിക്കുക (`docker run -d -p 6379:6379 redis`)
- ഒരു LLM API കീ (ഉദാഹരണം: OpenAI) — `.env` ൽ `LLM_API_KEY` സജ്ജമാക്കുക
- `.env` ൽ `CACHING=true` (Cognee സെഷനുകൾക്കായി ആവശ്യമാണ്)
- Azure AI Foundry പ്രോജക്ടും നൽകിയ ചാറ്റ് മോഡലും
- `.env` ൽ `AZURE_AI_PROJECT_ENDPOINT` ഉം `AZURE_AI_MODEL_DEPLOYMENT_NAME` ഉം
- Azure CLI ഉം ലോഗിൻ ചെയ്‌തിരിക്കുക (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ഏജന്റ് മെമ്മറിയുടെ തരം

പ്രധാനമായ ലെഷൻ 13 നോട്ട്ബുക്കിൽ നിന്ന് സമാനമായ മൂന്നു മെമ്മറി തരം ഈ നോട്ട്ബുക്ക് പരിശോധിക്കുന്നു, എന്നാൽ ദീർഘകാല മെമ്മറി ബാക്ക്എൻഡായി Cognee ഉപയോഗിക്കുന്നു:

| മെമ്മറി തരം | മെക്കാനിസം | ആയുസ്സ് |
|---|---|---|
| **വർക്കിംഗ്** | `agent.create_session()` (MAF) | ഏകസംവാദം |
| **ഹൃസ്വകാലം** | Cognee സെഷൻ കാഷെ (Redis) | ഏക സെഷൻ |
| **ദീർഘകാലം** | Cognee നോളേജ് ഗ്രാഫ് + വെക്ടറുകൾ | സ്ഥിരം |

### Cogneeയുടെ മെമ്മറി ആർക്കിടെക്ചർ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## കോഗ്നീ സംഭരണം തയ്യാറാക്കുക


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — നോളജ് ബേസ് നിർമ്മാണം

നമ്മുടെ കോഡിംഗ് അസിസ്റ്റന്റിനായി സമഗ്രമായ നോളജ് ബേസ് സൃഷ്ടിക്കാൻ ഞങ്ങൾ മൂന്നു തരം ഡാറ്റ സ്വീകരിക്കുന്നു:

1. **ഡവലപ്പർ പ്രൊഫൈൽ** — വ്യക്തിപരമായ വിദഗ്‌ധതയും സാങ്കേതിക പശ്ചാത്തലവും  
2. **പൈതൺ മികച്ച പ്രാക്ടീസുകൾ** — പ്രായോഗിക മാർഗനിർദേശങ്ങളോടുകൂടെ പൈതണിന്റെ സെൻ  
3. **ചരിത്ര സംഭാഷണങ്ങൾ** — ഡവലപ്പർമാരും AI അസിസ്റ്റന്റുകളും ഇടയ്ക്കുള്ള മുൻ Q&A സെഷനുകൾ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## നോളജ് ഗ്രാഫ് കാഴ്ചവയ്‌ക്കുക

Cognee അതിൻറെ എന്റിറ്റികളും ബന്ധങ്ങളും എടുക്കുന്നത് ഒരു ഇടപഴകുന്ന HTML കാഴ്ച്ചയായി കാണിക്കാം.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## മെമ്മറി മെമ്പിഫൈ ഉപയോഗിച്ച് സമ്പന്നമാക്കുക

`memify()` ക്വാള ledge ഗ്രാഫ് വിശകലനം ചെയ്യുന്നു, ബുദ്ധിമുട്ടുള്ള നിയമങ്ങൾ സൃഷ്ടിക്കുന്നു — പാറ്റേണുകൾ, മികച്ച രീതി, ആശയങ്ങൾ തമ്മിലുള്ള ബന്ധങ്ങൾ കണ്ടെത്തുന്നു.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ഭാഗം 2 — Cognee ഉപകരണങ്ങളോടുള്ള MAF ഏജന്റ 

ഇപ്പോൾ നാം `@tool` ഫംഗ്ഷനുകൾ മുഖാന്തിരം Cognee-യുടെ നോളജ് ഗ്രാഫ് ക്വറി ചെയ്യാൻ കഴിയുന്ന ഒരു MAF ഏജന്റ് സൃഷ്ടിക്കുന്നു. ഈ വഴി ഏജന്റിന് ഗ്രാഫ്-അവഗാഹന സമാന്തര അവബോധ സേർച്ച്-ന്റെ സമ്പൂർണ ശേഷി ഉപയോഗിക്കാനും സെഷനുകളിലൂടെ സംഭാഷണ സാന്ദർഭ്യം നിലനിർ‍ത്താനും കഴിയും.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## സെഷനുകളോടുകൂടെ വർക്കിംഗ് മെമ്മറി

`AgentSession` (`agent.create_session()` വഴി സൃഷ്ടിക്കുന്നത്) ഒരു സെഷനിനുള്ളിൽ വർക്കിംഗ് മെമ്മറി നൽകുന്നു. ഏജന്റ് മുൻ സന്ദേശങ്ങളെ തിരിച്ചറിയാനും കോഗ്നിയുടെ ദീർഘകാല നോളജ് ഗ്രാഫിനെ ചോദിക്കാനും കഴിയും.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## പുതിയ സെഷൻ — ദീർഘകാല സ്‌മৃতি നിലനിൽക്കും

പുതിയ സെഷൻ ആരംഭിക്കുന്നത് വർക്കിംഗ് മെമ്മറി ക്ലിയർ ചെയ്യും, പക്ഷേ Cognee നോളജ് ഗ്രാഫ് അതിനോടൊപ്പം ലഭ്യമാണ്. ഏജന്റ് ഒരു പൂർണ്ണമായും പുതിയ സംഭാഷണത്തിൽ അതേ ദീർഘകാല നോളേജ് പുനർപ്രാപിച്ചു വാങ്ങാൻ കഴിയും.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## സംക്ഷേപം

ഈ നോട്ട്ബുക്കിൽ നിങ്ങൾ നിർമ്മിച്ചത് കോഡിംഗ് അസിസ്റ്റന്റാണ്, ഇത് **MAFയുടെ പ്രവർത്തന മെമ്മറി** (`agent.create_session()`) **Cogneeയുടെ ദീര്‍ഘകാല ജ്ഞാനഗ്രാഫ്** എന്നിവ സംയോജിപ്പിക്കുന്നു.

### നിങ്ങൾ പഠിച്ചത്
1. **ജ്ഞാനഗ്രാഫ് നിർമാണം**: Cognee ഘടിതരായിട്ടില്ലാത്ത വാചകങ്ങൾ സ്വീകരിച്ച് ഗ്രാഫും വക്ടർ മെമ്മറിയും നിർമ്മിക്കുന്നു.
2. **memify ഉപയോഗിച്ച് ഗ്രാഫ് സമൃദ്ധീകരണം**: നിലവിലുള്ള ഗ്രാഫിന്റെ മേൽനോട്ടത്തിലുള്ള തമിഴിലുകൾക്കും സമൃദ്ധമായ ബന്ധങ്ങൾക്കും കാരണം നിൽക്കുന്നു.
3. **MAF + Cognee സംയോജനം**: `@tool` ഫങ്ക്ഷൻമാർ MAF ഏജന്റിന് Cogneeയുടെ ഗ്രാഫിൽ സ്വാഭാവികമായി ചോദിക്കാനാകും.
4. **പ്രവർത്തന മെമ്മറി + ദീര്‍ഘകാല മെമ്മറി**: `AgentSession` (`agent.create_session()` വഴി) സെഷൻ കണ്ടക്സ്റ്റ് നൽകുന്നു, Cognee സ്ഥിരതയുള്ള ജ്ഞാനം നൽകുന്നു.
5. **NodeSets ഉപയോഗിച്ച് ഫിൽറ്റർ ചെയ്ത തെരയൽ**: ജ്ഞാനഗ്രാഫിന്റെ പ്രത്യേക ഉപസമുഹങ്ങളെ ലക്ഷ്യം വെക്കാം (ഉദാ: നിയമങ്ങൾ മാത്രമേ).

### പ്രധാന പഠനങ്ങൾ
- **Cognee** കچے വാചകങ്ങളെ ഘടിതമായ, ബന്ധം ബോധമുള്ള മെമ്മറിയായി പരിവർത്തനം ചെയ്യുന്നു — ഫ്‌ളാറ്റ് വക്ടർ സ്റ്റോർ보다 കൂടുതൽ ശക്തമായത്.
- **`@tool` ഫംഗ്ഷൻമാർ** MAF ഏജന്റുകളും വാന്ത ആഭ്യന്തര ജ്ഞാന സംവിധാനങ്ങളും വൃത്തിയോടെ ബന്ധിപ്പിക്കുന്നു.
- **`AgentSession`** (`agent.create_session()` വഴി) സംഭാഷണാനുസര馍ത്തിനുള്ള കണ്ടക്സ്റ്റ് ദീര്‍ഘകാല ജ്ഞാനത്തിൽ നിന്നും വേർതിരിക്കുന്നു.
- ഒരേ ജ്ഞാനഗ്രാഫ് പല സെഷനുകൾക്കും ഏജന്റുകൾക്കും സേവനം ചെയ്യുന്നു.

### യഥാർത്ഥ ലോക ഉപയോഗങ്ങൾ
- **ഡെവലപ്പർ കോപിലോട്ടുകൾ**: കോഡ് പരിശോധന, ഇൻസിഡന്റ് വിശകലനം, ആർക്കിടെക്ചർ അസിസ്റ്റന്റുകൾ
- **ഉപഭോക്തൃ മുഖാമുഖ കോപിലോട്ടുകൾ**: ഉൽപ്പന്ന ഡോക്യൂമെന്റുകൾ, FAQകൾ, CRM കുറിപ്പുകൾ എന്നിവ വഴി സഹായ സംഘങ്ങൾ
- **അന്തരീക്ഷ വിദഗ്ധ കോപിലോട്ടുകൾ**: നയം, നിയമം, അല്ലെങ്കിൽ സുരക്ഷാ അസിസ്റ്റന്റുകൾ മാർഗ്ഗനിർദേശങ്ങൾ അടിസ്ഥാനമാക്കി നിഗമനങ്ങൾ വയ്ക്കുന്നു
- **ഏകീകൃത ഡാറ്റ പാളികൾ**: ഘടിതവും ഘടിതരായിട്ടില്ലാത്തവുമായ ഡാറ്റയെ ഒറ്റ ചോദ്യം ചെയ്യാവുന്ന ഗ്രാഫായി സംയോജിപ്പിക്കൽ

### അടുത്ത നടപടികൾ
- Cogneeയിൽ കാലാനുസൃത ബോധം പരീക്ഷിക്കുക
- മേഖലാ പ്രത്യേക ഗ്രാഫ് ഗുണനിലവാരത്തിനായി OWL ഒന്റോളജി നിർവ്വചിക്കുക
- തിരയൽ മെച്ചപ്പെടുത്താൻ ഉപയോക്തൃ പ്രതികരണ ചില്ലറകൾ ചേർക്കുക
- ഒരേ Cognee മെമ്മറി ലെയർ പങ്കിട്ടു പ്രവർത്തിക്കുന്ന മൾട്ടി-ഏജന്റ് സംവിധാനങ്ങളിലേക്ക് വ്യാപിപ്പിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**വിമുക്തികുറിപ്പ്**:  
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നിസ്സംശയം നമുക്ക് ശരീരത സംബന്ധിച്ച് ശ്രമിക്കുമ്പോഴും, സ്വയം പ്രവർത്തിക്കുന്ന പരിഭാഷകളിൽ പിശകുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടായേക്കാമെന്ന് ദയവായി മനസിലാക്കുക. ഈ ഡോക്യുമെന്റ് തൻറെ സ്വന്തം ഭാഷയിൽ ഉള്ള അസലായ രേഖയെയാണ് അധികാരപരമായ ഉറവിടമായി കണക്കാക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രഫഷണൽ മനുഷ്യ വിവർത്തനം നിർദ്ദേശിക്കപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിനാൽ ഉണ്ടായേക്കാവുന്ന തെറ്റിദ്ധാരണകൾക്കോ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കോ ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
